# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasist.structdata import detect_outliers
import os
from pathlib import Path

# 2. Data Loading and Understanding

In [2]:
pd.set_option("display.max_columns",None)

In [3]:
# Determining the data path
BASE_DIR = Path.cwd()
while not (BASE_DIR / "dataset").exists():
    BASE_DIR = BASE_DIR.parent
data_path = BASE_DIR / "dataset" / "house_prices.csv"

house_prices_df = pd.read_csv(data_path)

In [4]:
house_prices_df.shape

(187531, 21)

In [5]:
house_prices_df.head(5)

,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,Furnishing,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area
0,0,1 BHK Ready to Occupy Flat for sale in Srushti...,"Bhiwandi, Thane has an attractive 1 BHK Flat f...",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,Unfurnished,NaN,NaN,Srushti Siddhi Mangal Murti Complex,1,2,NaN,NaN,NaN,NaN,NaN
1,1,2 BHK Ready to Occupy Flat for sale in Dosti V...,One can find this stunning 2 BHK flat for sale...,98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,Semi-Furnished,East,Garden/Park,Dosti Vihar,2,NaN,1 Open,Freehold,NaN,NaN,NaN
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise...,Up for immediate sale is a 2 BHK apartment in ...,1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,Unfurnished,East,Garden/Park,Sunrise by Kalpataru,2,NaN,1 Covered,Freehold,NaN,NaN,NaN
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,This beautiful 1 BHK Flat is available for sal...,25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,Unfurnished,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
4,4,2 BHK Ready to Occupy Flat for sale in TenX Ha...,"This lovely 2 BHK Flat in Pokhran Road, Thane ...",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,Unfurnished,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2,NaN,1 Covered,Co-operative Society,NaN,NaN,NaN


In [6]:
house_prices_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 187531 entries, 0 to 187530
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Index              187531 non-null  int64  
 1   Title              187531 non-null  object 
 2   Description        184508 non-null  object 
 3   Amount(in rupees)  187531 non-null  object 
 4   Price (in rupees)  169866 non-null  float64
 5   location           187531 non-null  object 
 6   Carpet Area        106858 non-null  object 
 7   Status             186916 non-null  object 
 8   Floor              180454 non-null  object 
 9   Transaction        187448 non-null  object 
 10  Furnishing         184634 non-null  object 
 11  facing             117298 non-null  object 
 12  overlooking        106095 non-null  object 
 13  Society            77853 non-null   object 
 14  Bathroom           186703 non-null  object 
 15  Balcony            138596 non-null  object 
 16  Ca

# 3. Data Cleaning

## 3.1. Check & drop duplicat

In [7]:
# Check duplicate befor drop Index column
house_prices_df.duplicated().sum()

np.int64(0)

In [8]:
# dropping (Index, Dimensions, Plot Area) columns as this is not required for modelling
house_prices_df.drop(columns=["Index", "Dimensions", "Plot Area"],inplace=True)

In [9]:
# Check duplicate
house_prices_df.duplicated().sum()

np.int64(119339)

In [10]:
# drop duplicate
house_prices_df.drop_duplicates(inplace=True)

In [11]:
house_prices_df.shape

(68192, 18)

In [12]:
house_prices_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68192 entries, 0 to 187530
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Title              68192 non-null  object 
 1   Description        65974 non-null  object 
 2   Amount(in rupees)  68192 non-null  object 
 3   Price (in rupees)  62298 non-null  float64
 4   location           68192 non-null  object 
 5   Carpet Area        38075 non-null  object 
 6   Status             67893 non-null  object 
 7   Floor              65577 non-null  object 
 8   Transaction        68114 non-null  object 
 9   Furnishing         66902 non-null  object 
 10  facing             41392 non-null  object 
 11  overlooking        38657 non-null  object 
 12  Society            29372 non-null  object 
 13  Bathroom           67721 non-null  object 
 14  Balcony            49133 non-null  object 
 15  Car Parking        28863 non-null  object 
 16  Ownership          43264 n

## 3.2. Repairing the type of numerical columns 


In [13]:
# Repairing the type of numerical columns ["Amount(in rupees)", "Carpet Area", "Bathroom","Balcony","Super Area"]
num_columns = ["Amount(in rupees)", "Carpet Area", "Bathroom","Balcony","Super Area"]
for col in num_columns:
    print(f"{col} : {house_prices_df[col].unique()}")
    print("-"*50)

Amount(in rupees) : ['42 Lac ' '98 Lac ' '1.40 Cr ' ... '1.5 Lac ' '24.4 Lac ' '9.90 Cr ']
--------------------------------------------------
Carpet Area : ['500 sqft' '473 sqft' '779 sqft' ... '1634 sqft' '164 sqyrd' '136 sqft']
--------------------------------------------------
Bathroom : ['1' '2' '3' '4' '6' nan '5' '10' '9' '8' '> 10' '7']
--------------------------------------------------
Balcony : ['2' nan '1' '3' '4' '> 10' '6' '5' '7' '10' '8' '9']
--------------------------------------------------
Super Area : [nan '680 sqft' '575 sqft' ... '2066 sqft' '406 sqft' '2332 sqft']
--------------------------------------------------


### 1. Amount(in rupees)

In [14]:
# Detecting incorrect values ​​in Amount(in rupees) column 
dirty_values = []
for value in house_prices_df["Amount(in rupees)"].unique().tolist():
    value_split = str(value).split()
    for i in value_split:
        try:
            float(i)
        except:
            if i not in dirty_values:

                dirty_values.append(i)
            else:
                continue

dirty_values

['Lac', 'Cr', 'Call', 'for', 'Price']

In [15]:
# convert Amount(in rupees) column to numeric
def convert_amount_to_numeric(value):
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).lower().strip()
    value_split = value_str.split()
    
    if not value_split or "call" in value_split or "price" in value_split:
        return np.nan
    
    for i in value_split:

        try:
            num = float(i)
            
            if "cr" in value_split:
                return num * 10000000
            elif "lac" in value_split:
                return num * 100000
            else:
                return num 
                
        except ValueError:
            continue
    
    return np.nan

house_prices_df['Amount(in rupees)'] = house_prices_df['Amount(in rupees)'].apply(convert_amount_to_numeric)

### 2. Carpet Area

In [16]:
# Detecting incorrect values ​​in Carpet Area column 
dirty_values = []
for value in house_prices_df["Carpet Area"].unique().tolist():
    value_split = str(value).split()
    for i in value_split:
        try:
            float(i)
        except:
            if i not in dirty_values:

                dirty_values.append(i)
            else:
                continue

dirty_values

['sqft', 'sqm', 'sqyrd', 'acre', 'ground', 'cent', 'bigha', 'marla', 'kanal']

In [17]:
# convert area to sqft
def convert_area_to_sqft(value):
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).lower().strip()
    value_split = value_str.split()

    if not value_split:
        return np.nan

    for i in value_split:

        try:
            num = float(i.replace(",",""))  

            if "sqm" in value_str:
                return num * 10.764
            elif "sqyrd" in value_str:
                return num * 9
            elif "acre" in value_str:
                return num * 43560
            elif "ground" in value_str:
                return num * 2400
            elif "cent" in value_str:
                return num * 435.6
            elif "bigha" in value_str:
                return num * 27000
            elif "marla" in value_str:
                return num * 272.25
            elif "kanal" in value_str:
                return num * 5445
            else:
                return num
        
        except:
            continue
    return np.nan

house_prices_df['Carpet Area'] = house_prices_df['Carpet Area'].apply(convert_area_to_sqft)

### 3. Super Area

In [18]:
# Detecting incorrect values ​​in Super Area column 
dirty_values = []
for value in house_prices_df["Super Area"].unique().tolist():
    value_split = str(value).split()
    for i in value_split:
        try:
            float(i)
        except:
            if i not in dirty_values:

                dirty_values.append(i)
            else:
                continue

dirty_values

['sqft',
 '10,000',
 'sqm',
 '15,000',
 '12,000',
 'sqyrd',
 '11,000',
 'marla',
 '13,500',
 '40,000',
 '36,000',
 '20,000',
 '10,019',
 '10,590',
 '13,200',
 '25,000',
 'kanal',
 'ground',
 '12,600',
 'biswa2',
 'aankadam',
 'acre',
 '37,952',
 '18,000',
 'hectare',
 'cent',
 '530,040',
 '13,000']

In [19]:
# convert super area to sqft
def convert_super_area_to_sqft(value):
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).strip().lower()
    value_split = value_str.split()
    
    if not value_split:
        return np.nan
    
    for i in value_split:
        try:
            num = float(i.replace(',', '').strip())
            
            if "sqm" in value_str:
                return num * 10.764
            elif "sqyrd" in value_str:
                return num * 9
            elif "acre" in value_str:
                return num * 43560
            elif "hectare" in value_str:
                return num * 107639
            elif "ground" in value_str:
                return num * 2400
            elif "cent" in value_str:
                return num * 435.6
            elif "bigha" in value_str:
                return num * 27000
            elif "marla" in value_str:
                return num * 272.25
            elif "kanal" in value_str:
                return num * 5445
            elif "biswa" in value_str:
                return num * 1350
            elif "aankadam" in value_str:
                return num * 72
            else:
                
                return num
                
        except :
            continue
            
    return np.nan

house_prices_df['Super Area'] = house_prices_df['Super Area'].apply(convert_super_area_to_sqft)

### 4. Bathroom

In [20]:
# Detecting incorrect values ​​in Bathroom column 
dirty_values = []
for value in house_prices_df["Bathroom"].unique().tolist():
    value_split = str(value).split()
    for i in value_split:
        try:
            float(i)
        except:
            if i not in dirty_values:

                dirty_values.append(i)
            else:
                continue

dirty_values

['>']

In [21]:
# convert bathroom to numeric
def convert_bathroom_to_numeric(value):
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).strip()
    
    if value_str == "> 10":
        return 10
        
    try:
        return int(float(value_str))
    except ValueError:
        return np.nan

house_prices_df['Bathroom'] = house_prices_df['Bathroom'].apply(convert_bathroom_to_numeric)

In [22]:
house_prices_df["Bathroom"].unique()

array([ 1.,  2.,  3.,  4.,  6., nan,  5., 10.,  9.,  8.,  7.])

### 5. Balcony

In [23]:
def convert_Balcony_to_numeric(value):
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).strip()
    
    if value_str == "> 10":
        return 10
        
    try:
        return int(float(value_str))
    except ValueError:
        return np.nan

house_prices_df['Balcony'] = house_prices_df['Balcony'].apply(convert_Balcony_to_numeric)

## 3.3. Feature Engineering

### 1. BHK

In [24]:
# Extract Number Of BHK from Title column
house_prices_df['BHK'] = house_prices_df['Title'].str.lower().str.extract(r'(\d+)\s*bhk').astype("Int64")

### 2. Current Floor & Total_Floors

In [25]:
# Extracting the Current Floor & Total_Floors from the floor column

floor_data = house_prices_df['Floor'].str.extract(r'(\w+)\s+out\s+of\s+(\d+)')
floor_data.columns = ['Current_Floor', 'Total_Floors']

floor_data['Current_Floor'] = floor_data['Current_Floor'].str.strip().str.capitalize()

floor_mapping = {
    'Ground': 0,
    'Basement': -1,
    'Lower': -1,
    'Upper': 0
}

floor_data['Current_Floor'] = floor_data['Current_Floor'].replace(floor_mapping)
floor_data['Current_Floor'] = pd.to_numeric(floor_data['Current_Floor'], errors='coerce').astype("Int64")

floor_data['Total_Floors'] = pd.to_numeric(floor_data['Total_Floors'], errors='coerce').astype("Int64")

house_prices_df['Current_Floor'] = floor_data['Current_Floor']
house_prices_df['Total_Floors'] = floor_data['Total_Floors']

In [26]:
# Delete the incorrect values
error_indices = floor_data[floor_data['Current_Floor'] > floor_data['Total_Floors']].index

house_prices_df = house_prices_df.drop(error_indices)

### 3. [View_Garden_Park , View_Main_Road, View_Pool, View_Unknown]

In [27]:
# Extract information from the overlooking column and delete it 
house_prices_df['overlooking'] = house_prices_df['overlooking'].fillna('Unknown')

house_prices_df['View_Garden_Park'] = house_prices_df['overlooking'].str.contains('Garden/Park', case=False).astype(int)
house_prices_df['View_Main_Road']  = house_prices_df['overlooking'].str.contains('Main Road', case=False).astype(int)
house_prices_df['View_Pool']       = house_prices_df['overlooking'].str.contains('Pool', case=False).astype(int)

house_prices_df['View_Unknown'] = (
    (house_prices_df['View_Garden_Park'] == 0) &
    (house_prices_df['View_Main_Road'] == 0) &
    (house_prices_df['View_Pool'] == 0)
).astype(int)

house_prices_df.drop(columns=['overlooking'], inplace=True)

In [28]:
house_prices_df.shape

(68186, 24)

## 3.4. Handling Missing Values

In [29]:
#calculate percentage of missing values
(house_prices_df.isnull().mean()) * 100

Title                 0.000000
Description           3.252867
Amount(in rupees)     4.302936
Price (in rupees)     8.638137
location              0.000000
Carpet Area          44.161558
Status                0.438506
Floor                 3.835098
Transaction           0.114393
Furnishing            1.890417
facing               39.295457
Society              56.926642
Bathroom              0.690758
Balcony              27.950019
Car Parking          57.670196
Ownership            36.550025
Super Area           56.017364
BHK                   0.476637
Current_Floor         3.905494
Total_Floors          3.905494
View_Garden_Park      0.000000
View_Main_Road        0.000000
View_Pool             0.000000
View_Unknown          0.000000
dtype: float64

### Super Area

In [30]:
# Calculating missing values ​​in Super Area column 
missing_Super_Area = house_prices_df[house_prices_df["Super Area"].isnull()]["Super Area"].index
house_prices_df.loc[missing_Super_Area, "Super Area"] = house_prices_df.loc[missing_Super_Area, "Amount(in rupees)"] / house_prices_df.loc[missing_Super_Area, "Price (in rupees)"]
house_prices_df['Super Area'] = house_prices_df['Super Area'].replace([np.inf, -np.inf], np.nan)
house_prices_df.dropna(subset=['Super Area'], inplace=True)
house_prices_df = house_prices_df[house_prices_df['Super Area'] >= 10]

In [31]:
# Delete the small percentage of missing values
columns_to_dropna = ['Furnishing', 'Status', 'Transaction',"Bathroom","BHK","Amount(in rupees)","Floor","Current_Floor","Total_Floors","Super Area"]
house_prices_df.dropna(subset=columns_to_dropna,inplace=True)

### facing & Ownership

In [32]:
# Fill  the missing values ​​in the facing & Ownership columns with Unknown
house_prices_df["facing"] = house_prices_df["facing"].fillna('Unknown')
house_prices_df['Ownership'] = house_prices_df['Ownership'].fillna('Unknown')

### Balcony

In [33]:
# Fill  the missing values ​​in Balcony column with 0
house_prices_df['Balcony'] = house_prices_df['Balcony'].fillna(0).astype(int)

In [34]:
# Delete unnecessary columns
columns_to_drop = ['Title', 'Description', 'Carpet Area',"Car Parking", "Society","Floor"]
house_prices_df.drop(columns=columns_to_drop, inplace=True)

In [35]:
# Check Missing Values
house_prices_df.isnull().sum()

Amount(in rupees)    0
Price (in rupees)    0
location             0
Status               0
Transaction          0
Furnishing           0
facing               0
Bathroom             0
Balcony              0
Ownership            0
Super Area           0
BHK                  0
Current_Floor        0
Total_Floors         0
View_Garden_Park     0
View_Main_Road       0
View_Pool            0
View_Unknown         0
dtype: int64

In [36]:
# Check Shape
house_prices_df.shape

(58567, 18)

In [37]:
house_prices_df['Bathroom'] = house_prices_df['Bathroom'].astype("Int64")

## 3.5. Handling Outliers

In [38]:
# detect all outliers
indices = detect_outliers(data=house_prices_df, n=0, features=house_prices_df.select_dtypes(include="number"))
len(indices)

17868

In [39]:
# Calculating outliers
def detect_outliers_report(df, column_name):
    Q1 = df[column_name].quantile(0.25)
    Q3 = df[column_name].quantile(0.75)
    IQR = Q3 - Q1

    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    outliers_upper_df = df[df[column_name] > upper_fence]
    outliers_lower_df = df[df[column_name] < lower_fence]
    
    outlier_indices = outliers_upper_df.index.union(outliers_lower_df.index).tolist()
    
    total_outliers = len(outlier_indices)
    total_rows = len(df)
    outliers_percentage = (total_outliers / total_rows) * 100

    # 4. Print English Report
    print(f"=== OUTLIER DETECTION REPORT FOR: [{column_name}] ===")
    print(f"Q1 (25th Percentile) : {Q1:,.2f}")
    print(f"Q3 (75th Percentile) : {Q3:,.2f}")
    print(f"IQR                  : {IQR:,.2f}")
    print(f"Lower Fence          : {lower_fence:,.2f}")
    print(f"Upper Fence          : {upper_fence:,.2f}")
    print("-" * 50)
    print(f"⚠️ Upper Outliers Count : {len(outliers_upper_df)}")
    print(f"⚠️ Lower Outliers Count : {len(outliers_lower_df)}")
    print(f"📊 Total Outliers Found  : {total_outliers} out of {total_rows} rows ({outliers_percentage:.2f}%)")
    print("=" * 50 + "\n")

    return outlier_indices

In [40]:
# Calculating outliers for each numerical column
numerical_cols = ["Amount(in rupees)","Price (in rupees)", "Super Area", "Bathroom", "Balcony", "BHK", "Current_Floor", "Total_Floors"]
all_outliers = []
for col in numerical_cols:
    outlier_indices = detect_outliers_report(house_prices_df,col)
    all_outliers.append(outlier_indices)

=== OUTLIER DETECTION REPORT FOR: [Amount(in rupees)] ===
Q1 (25th Percentile) : 4,200,000.00
Q3 (75th Percentile) : 11,000,000.00
IQR                  : 6,800,000.00
Lower Fence          : -6,000,000.00
Upper Fence          : 21,200,000.00
--------------------------------------------------
⚠️ Upper Outliers Count : 5658
⚠️ Lower Outliers Count : 0
📊 Total Outliers Found  : 5658 out of 58567 rows (9.66%)

=== OUTLIER DETECTION REPORT FOR: [Price (in rupees)] ===
Q1 (25th Percentile) : 3,828.00
Q3 (75th Percentile) : 7,119.50
IQR                  : 3,291.50
Lower Fence          : -1,109.25
Upper Fence          : 12,056.75
--------------------------------------------------
⚠️ Upper Outliers Count : 4360
⚠️ Lower Outliers Count : 0
📊 Total Outliers Found  : 4360 out of 58567 rows (7.44%)

=== OUTLIER DETECTION REPORT FOR: [Super Area] ===
Q1 (25th Percentile) : 1,000.00
Q3 (75th Percentile) : 1,743.00
IQR                  : 743.00
Lower Fence          : -114.50
Upper Fence          : 2,85

In [41]:
# Check describe befor handling outliers
house_prices_df.describe()

,Amount(in rupees),Price (in rupees),Bathroom,Balcony,Super Area,BHK,Current_Floor,Total_Floors,View_Garden_Park,View_Main_Road,View_Pool,View_Unknown
count,5.856700e+04,5.856700e+04,58567.0,58567.000000,5.856700e+04,58567.0,58567.0,58567.0,58567.000000,58567.000000,58567.000000,58567.000000
mean,1.085220e+07,6.674246e+03,2.437055,1.589223,2.697174e+03,2.567726,4.416531,8.745283,0.410094,0.444448,0.142111,0.392969
std,6.616343e+07,3.767697e+04,0.886112,1.266570,1.924178e+05,0.83775,4.504703,7.021102,0.491855,0.496909,0.349167,0.488414
min,1.000000e+05,0.000000e+00,1.0,0.000000,1.100001e+01,1.0,-1.0,1.0,0.000000,0.000000,0.000000,0.000000
25%,4.200000e+06,3.828000e+03,2.0,1.000000,1.000000e+03,2.0,2.0,4.0,0.000000,0.000000,0.000000,0.000000
50%,6.500000e+06,5.060000e+03,2.0,2.000000,1.299907e+03,3.0,3.0,6.0,0.000000,0.000000,0.000000,0.000000
75%,1.100000e+07,7.119500e+03,3.0,2.000000,1.742997e+03,3.0,6.0,12.0,1.000000,1.000000,0.000000,1.000000
max,1.400300e+10,6.700000e+06,10.0,10.000000,4.347288e+07,10.0,200.0,200.0,1.000000,1.000000,1.000000,1.000000


In [42]:
# handling outliers
def location_outlier_handler(df,target_column):
    cleaned_parts = []

    for location , subdf in df.groupby("location"):
        if len(subdf) < 5:
            cleaned_parts.append(subdf)
            continue

        skewness = subdf[target_column].skew()
        mean = subdf[target_column].mean()
        std = subdf[target_column].std()

        Q1 = subdf[target_column].quantile(0.25)
        Q3 = subdf[target_column].quantile(0.75)
        IQR = Q3 - Q1

        if -0.5 <= skewness <= 0.5:
            lower_fence = mean - 2 * std
            upper_fence = mean + 2 * std
        else:
            lower_fence = Q1 - 1.5 * IQR
            upper_fence = Q3 + 1.5 * IQR

        filtered_subdf = subdf[(subdf[target_column] >= lower_fence) & (subdf[target_column] <= upper_fence)]

        cleaned_parts.append(filtered_subdf)

    final_df = pd.concat(cleaned_parts).reset_index(drop=True)
    return final_df

### 1_ Price (in rupees)

In [43]:
clean_house_prices_df = location_outlier_handler(house_prices_df, "Price (in rupees)")
print(f"old shape : {house_prices_df.shape}")
print(f"new shape : {clean_house_prices_df.shape}")
clean_house_prices_df.describe()

old shape : (58567, 18)
new shape : (56097, 18)


,Amount(in rupees),Price (in rupees),Bathroom,Balcony,Super Area,BHK,Current_Floor,Total_Floors,View_Garden_Park,View_Main_Road,View_Pool,View_Unknown
count,5.609700e+04,56097.000000,56097.0,56097.000000,56097.000000,56097.0,56097.0,56097.0,56097.000000,56097.000000,56097.000000,56097.000000
mean,9.369531e+06,5955.262296,2.406688,1.574844,1442.149450,2.542311,4.372426,8.655329,0.408293,0.443624,0.140132,0.395583
std,1.168563e+07,3740.246054,0.858643,1.255436,829.234650,0.819575,4.421954,6.848194,0.491522,0.496816,0.347127,0.488980
min,2.000000e+05,279.000000,1.0,0.000000,55.000000,1.0,-1.0,1.0,0.000000,0.000000,0.000000,0.000000
25%,4.050000e+06,3793.000000,2.0,1.000000,1000.000000,2.0,2.0,4.0,0.000000,0.000000,0.000000,0.000000
50%,6.400000e+06,5000.000000,2.0,2.000000,1280.000000,3.0,3.0,6.0,0.000000,0.000000,0.000000,0.000000
75%,1.000000e+07,6785.000000,3.0,2.000000,1704.822211,3.0,5.0,12.0,1.000000,1.000000,0.000000,1.000000
max,4.200000e+08,46250.000000,10.0,10.000000,36000.000000,10.0,200.0,200.0,1.000000,1.000000,1.000000,1.000000


#### 🧹 1. Price (in rupees) Outlier Handling Insights

* **Method Used:**
  We removed outliers using `groupby('location')`.
  This means we compare each property's price with other properties in the *same location*, not with the whole dataset.
  This helps us be fair because prices are different from one area to another.

* **What Changed in the Data:**

  * We removed wrong or extreme values (like a price of **322,727**), and now the maximum is a more realistic **42,000** rupees/sq.ft.
  * The data was highly skewed to the right before, but now the **mean is closer to the median**, which makes the data more balanced.
  * We kept real expensive properties in rich areas, so we didn’t lose important information.

* **Why This is Good:**
  The dataset is now cleaner, more realistic, and better for building machine learning models.


### 2_ Super Area

In [44]:
final_clean_df = location_outlier_handler(clean_house_prices_df, "Super Area")
print(f"original shape : {house_prices_df.shape}")
print(f"old shape : {clean_house_prices_df.shape}")
print(f"new shape : {final_clean_df.shape}")
final_clean_df.describe()

original shape : (58567, 18)
old shape : (56097, 18)
new shape : (53490, 18)


,Amount(in rupees),Price (in rupees),Bathroom,Balcony,Super Area,BHK,Current_Floor,Total_Floors,View_Garden_Park,View_Main_Road,View_Pool,View_Unknown
count,5.349000e+04,53490.000000,53490.0,53490.000000,53490.000000,53490.0,53490.0,53490.0,53490.000000,53490.000000,53490.000000,53490.000000
mean,8.249690e+06,5837.985979,2.330884,1.543466,1337.733575,2.477734,4.262105,8.50172,0.402973,0.442382,0.136343,0.399570
std,7.540481e+06,3606.713764,0.758466,1.220616,538.822974,0.747508,4.159726,6.602268,0.490500,0.496674,0.343156,0.489815
min,2.000000e+05,279.000000,1.0,0.000000,55.000000,1.0,-1.0,1.0,0.000000,0.000000,0.000000,0.000000
25%,4.000000e+06,3753.250000,2.0,0.000000,980.000000,2.0,2.0,4.0,0.000000,0.000000,0.000000,0.000000
50%,6.100000e+06,4916.500000,2.0,2.000000,1250.000000,3.0,3.0,5.0,0.000000,0.000000,0.000000,0.000000
75%,9.500000e+06,6625.000000,3.0,2.000000,1649.862511,3.0,5.0,12.0,1.000000,1.000000,0.000000,1.000000
max,1.000000e+08,46250.000000,10.0,10.000000,4366.074349,10.0,40.0,91.0,1.000000,1.000000,1.000000,1.000000


#### 📐 2. Super Area Outlier Handling Insights

* **Method Used:**
  We used the same `location_outlier_handler` method.
  This means we cleaned the area values by comparing properties inside the *same location*, not the whole dataset.

* **What Changed in the Data:**

  * We removed very large and unrealistic values (like **36,000** sq.ft), and now the maximum is a more reasonable **4,366** sq.ft.
  * The **standard deviation (Std)** decreased from **819** to **535**, which means the data is now more consistent and less noisy.
  * Some related wrong values (like very high floors in `Current_Floor` and `Total_Floors`) were also removed automatically because they are connected to the area feature.

* **Why This is Good:**
  The data is now cleaner, more logical, and easier for the model to learn from.


In [45]:
# Check if there are any rows where the current floor is greater than the total floors
logical_error_df = final_clean_df[final_clean_df['Current_Floor'] > final_clean_df['Total_Floors']]

print("Number of rows with logical floor errors:", len(logical_error_df))

Number of rows with logical floor errors: 0


In [46]:
# Drop the 'Price (in rupees)' column to prevent data leakage
final_clean_df.drop(columns="Price (in rupees)", inplace=True)

# Check how many duplicate rows exist before removing them
num_duplicates = final_clean_df.duplicated().sum()
print(f"Number of duplicate rows before dropping: {num_duplicates}")

# Remove duplicate rows to ensure data consistency
final_clean_df.drop_duplicates(inplace=True)

# Reset the index after cleaning to keep it clean and sequential
final_clean_df.reset_index(drop=True, inplace=True)

Number of duplicate rows before dropping: 1142


* **Dropped `Price (in rupees)`:**
  We removed this column before feature engineering because our main target is the total price (`Amount (in rupees)`).
  Keeping it could cause **data leakage**, since:

  ```math
  Amount = Super Area × Price
  ```

  So the model might learn the answer directly instead of learning real patterns.


# 4. save cleaned data

In [47]:
# Save the cleaned dataset to a CSV file for later use (modeling / deployment)
final_clean_df.to_csv(r"..\dataset\clean_house_prices_df.csv",index=False)